# Predicao de Acoes Bancarias com LSTM + GRU (PyTorch)

**Ativos:** ITUB4, BBDC4, BBAS3, SANB11  
**Periodo:** 2022-01-01 a 2025-07-31  
**Split:** 80% treino / 20% teste (cronologico)  
**Features:** Close, SMA_14, EMA_14, RSI_14 (precos absolutos)  
**Modelo:** LSTM → Dropout → GRU → Linear(4)

---
## FASE 1 — Estudo do Objeto (EDA)

In [ ]:
# ── Instalacao e imports ───────────────────────────────────────
!pip install yfinance matplotlib seaborn scikit-learn torch --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import time, warnings
from copy import deepcopy
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', DEVICE)

In [ ]:
# ── Configuracao ───────────────────────────────────────────────
TICKERS    = ['ITUB4.SA', 'BBDC4.SA', 'BBAS3.SA', 'SANB11.SA']
NAMES      = ['Itau Unibanco', 'Bradesco', 'Banco do Brasil', 'Santander Brasil']
START      = '2022-01-01'
END        = '2025-07-31'
N_STOCKS   = 4
N_FEATURES = 16            # 4 ativos × 4 features
TARGET_COLS = [0, 4, 8, 12]  # coluna Close de cada ativo

In [ ]:
# ── Download e feature engineering ────────────────────────────
def add_indicators(series, name, window=14):
    df = pd.DataFrame({f'{name}_Close': series})
    col = f'{name}_Close'
    df[f'{name}_SMA_{window}'] = df[col].rolling(window).mean()
    df[f'{name}_EMA_{window}'] = df[col].ewm(span=window, adjust=False).mean()
    delta = df[col].diff()
    rs = delta.clip(lower=0).rolling(window).mean() / (-delta.clip(upper=0)).rolling(window).mean()
    df[f'{name}_RSI_{window}'] = 100 - (100 / (1 + rs))
    return df

print('Baixando dados...')
dfs = []
for ticker in TICKERS:
    for _ in range(3):
        raw = yf.download(ticker, start='2021-07-01', end=END, auto_adjust=True, progress=False)
        if len(raw) > 0: break
        time.sleep(3)
    dfs.append(add_indicators(raw['Close'].squeeze(), name=ticker))
    print(f'  {ticker}: {len(raw)} registros')

data = pd.concat(dfs, axis=1).dropna()
data = data[data.index >= START]
print(f'\nPeriodo: {data.index[0].date()} a {data.index[-1].date()} ({len(data)} pregoes)')
data.tail(3)

In [ ]:
# ── Split e visualizacao dos precos ───────────────────────────
split = int(0.8 * len(data))
train_data = data.iloc[:split]
test_data  = data.iloc[split:]
print(f'Treino: {train_data.index[0].date()} a {train_data.index[-1].date()} ({len(train_data)} dias)')
print(f'Teste : {test_data.index[0].date()} a {test_data.index[-1].date()} ({len(test_data)} dias)')

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
fig, axes = plt.subplots(2, 2, figsize=(14, 7))
axes = axes.flatten()
for i, (t, name) in enumerate(zip(TICKERS, NAMES)):
    col = f'{t}_Close'
    axes[i].plot(train_data.index, train_data[col], color=colors[i], lw=1.2, label='Treino (80%)')
    axes[i].plot(test_data.index,  test_data[col],  color=colors[i], lw=1.5, ls='--', label='Teste (20%)')
    axes[i].axvline(train_data.index[-1], color='gray', ls=':', lw=1)
    axes[i].set_title(f'{name} ({t})', fontweight='bold')
    axes[i].set_ylabel('Preco (R$)')
    axes[i].legend(fontsize=8); axes[i].grid(True, alpha=0.3)
plt.suptitle('Evolucao dos Precos — Treino vs Teste', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Correlacao entre precos ────────────────────────────────────
close_cols = [f'{t}_Close' for t in TICKERS]
corr = train_data[close_cols].rename(columns=dict(zip(close_cols, NAMES))).corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=0.5, annot_kws={'size': 11})
plt.title('Correlacao de Precos — Treino (80%)', fontsize=12)
plt.tight_layout(); plt.show()

print('Correlacao media por ativo:')
for i, name in enumerate(NAMES):
    outros = [corr.iloc[i,j] for j in range(4) if i != j]
    print(f'  {name:22s}: {np.mean(outros):.3f}')

---
## FASE 2 — Implementacao do Modelo LSTM + GRU

**Arquitetura:**
```
Entrada (batch, seq_len, 16)
   -> LSTM(hidden)   aprende padroes de longo prazo
   -> Dropout(0.2)   regularizacao
   -> GRU(hidden)    refina com eventos recentes
   -> Linear(4)      preve preco de fechamento dos 4 ativos
```

**LSTM — mecanismo de porta:**
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

**GRU — versao simplificada:** combina as portas de esquecimento e entrada em uma unica porta de atualizacao, resultando em menos parametros e treinamento mais rapido.

In [ ]:
# ── Normalizacao e sequencias ──────────────────────────────────
scaler = MinMaxScaler()
scaler.fit(train_data.values)
data_scaled = scaler.transform(data.values)

def make_loaders(seq_len, batch=32):
    # Treino: primeiros 'split' dias
    X_tr, y_tr = [], []
    for i in range(seq_len, split):
        X_tr.append(data_scaled[i-seq_len:i])
        y_tr.append(data_scaled[i, TARGET_COLS])
    X_tr, y_tr = np.array(X_tr), np.array(y_tr)

    # Teste: ultimos (len-split) dias com lookback
    X_te, y_te = [], []
    for i in range(split, len(data_scaled)):
        X_te.append(data_scaled[i-seq_len:i])
        y_te.append(data_scaled[i, TARGET_COLS])
    X_te, y_te = np.array(X_te), np.array(y_te)

    vs = int(0.8 * len(X_tr))  # 80% treino, 20% validacao
    class DS(Dataset):
        def __init__(self, X, y):
            self.X = torch.tensor(X, dtype=torch.float32)
            self.y = torch.tensor(y, dtype=torch.float32)
        def __len__(self): return len(self.X)
        def __getitem__(self, i): return self.X[i], self.y[i]

    tr  = DataLoader(DS(X_tr[:vs], y_tr[:vs]), batch_size=batch, shuffle=True)
    val = DataLoader(DS(X_tr[vs:], y_tr[vs:]), batch_size=batch)
    te  = DataLoader(DS(X_te, y_te), batch_size=batch)
    print(f'seq={seq_len} | treino={vs} | val={len(X_tr)-vs} | teste={len(X_te)}')
    return y_tr[vs:], y_te, tr, val, te

print('Scaler ajustado no treino. Sequencias prontas.')

In [ ]:
# ── Modelo LSTM + GRU ──────────────────────────────────────────
class LSTMGRUModel(nn.Module):
    def __init__(self, input_size, hidden_lstm=64, hidden_gru=32, dropout=0.2):
        super().__init__()
        self.lstm    = nn.LSTM(input_size, hidden_lstm, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.gru     = nn.GRU(hidden_lstm, hidden_gru, batch_first=True)
        self.fc      = nn.Linear(hidden_gru, N_STOCKS)  # 4 saidas

    def forward(self, x):
        out, _ = self.lstm(x)
        out    = self.dropout(out)
        out, _ = self.gru(out)
        return self.fc(out[:, -1, :])  # ultimo passo temporal

# ── Treinamento ────────────────────────────────────────────────
def treinar(model, tr, val, epochs=100, lr=0.001):
    crit = nn.MSELoss()
    opt  = torch.optim.Adam(model.parameters(), lr=lr)
    best_val, best_state = float('inf'), None
    tl_list, vl_list = [], []
    for ep in range(1, epochs+1):
        model.train()
        bl = []
        for xb, yb in tr:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward(); opt.step()
            bl.append(loss.item())
        model.eval()
        vl = []
        with torch.no_grad():
            for xb, yb in val:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                vl.append(crit(model(xb), yb).item())
        tl, vl_m = np.mean(bl), np.mean(vl)
        tl_list.append(tl); vl_list.append(vl_m)
        if vl_m < best_val:
            best_val = vl_m; best_state = deepcopy(model.state_dict())
        if ep % 10 == 0:
            print(f'  Ep {ep:3d} | Train={tl:.5f} | Val={vl_m:.5f}')
    model.load_state_dict(best_state)
    return model, tl_list, vl_list

# ── Avaliacao ──────────────────────────────────────────────────
def inv(vals, col):
    buf = np.zeros((len(vals), N_FEATURES))
    buf[:, col] = vals
    return scaler.inverse_transform(buf)[:, col]

def avaliar(model, te, y_te_norm):
    model.eval(); preds = []
    with torch.no_grad():
        for xb, _ in te:
            preds.extend(model(xb.to(DEVICE)).cpu().numpy())
    preds = np.array(preds)
    rows = {}
    for i, (ci, name) in enumerate(zip(TARGET_COLS, NAMES)):
        p = inv(preds[:, i], ci)
        r = inv(y_te_norm[:, i], ci)
        rows[name] = {
            'RMSE (R$)': round(np.sqrt(mean_squared_error(r, p)), 3),
            'MAE (R$)':  round(mean_absolute_error(r, p), 3),
            'R2':        round(r2_score(r, p), 4),
            'MAPE (%)':  round(np.mean(np.abs((r-p)/r))*100, 2),
        }
    return pd.DataFrame(rows).T, preds

print('Modelo e funcoes definidos.')

In [ ]:
# ── Baseline: LSTM=64, GRU=32, seq=60, epochs=100 ─────────────
print('=== BASELINE: LSTM=64, GRU=32, seq=60 ===')
_, y_te1, tr1, val1, te1 = make_loaders(60)
m1 = LSTMGRUModel(N_FEATURES, 64, 32).to(DEVICE)
print(f'Parametros: {sum(p.numel() for p in m1.parameters()):,}')
m1, tl1, vl1 = treinar(m1, tr1, val1, epochs=100)
met1, preds1 = avaliar(m1, te1, y_te1)
print(met1.to_string())

In [ ]:
# ── Exp.2: LSTM=128, GRU=64, seq=60, epochs=100 ───────────────
print('=== EXP.2: LSTM=128, GRU=64, seq=60 ===')
_, y_te2, tr2, val2, te2 = make_loaders(60)
m2 = LSTMGRUModel(N_FEATURES, 128, 64).to(DEVICE)
print(f'Parametros: {sum(p.numel() for p in m2.parameters()):,}')
m2, tl2, vl2 = treinar(m2, tr2, val2, epochs=100)
met2, preds2 = avaliar(m2, te2, y_te2)
print(met2.to_string())

In [ ]:
# ── Exp.3: LSTM=128, GRU=64, seq=90, epochs=100 ───────────────
print('=== EXP.3: LSTM=128, GRU=64, seq=90 ===')
_, y_te3, tr3, val3, te3 = make_loaders(90)
m3 = LSTMGRUModel(N_FEATURES, 128, 64).to(DEVICE)
print(f'Parametros: {sum(p.numel() for p in m3.parameters()):,}')
m3, tl3, vl3 = treinar(m3, tr3, val3, epochs=100)
met3, preds3 = avaliar(m3, te3, y_te3)
print(met3.to_string())

---
## FASE 3 — Validacao e Resultados

In [ ]:
# ── Tabela comparativa ─────────────────────────────────────────
comp = pd.DataFrame({
    'Baseline (L=64, G=32, seq=60)' : [met1['RMSE (R$)'].mean(), met1['R2'].mean(), met1['MAPE (%)'].mean()],
    'Exp.2   (L=128, G=64, seq=60)' : [met2['RMSE (R$)'].mean(), met2['R2'].mean(), met2['MAPE (%)'].mean()],
    'Exp.3   (L=128, G=64, seq=90)' : [met3['RMSE (R$)'].mean(), met3['R2'].mean(), met3['MAPE (%)'].mean()],
}, index=['RMSE medio (R$)', 'R2 medio', 'MAPE medio (%)']).round(4).T

print('=== COMPARACAO DOS 3 EXPERIMENTOS ===')
print(comp.to_string())

In [ ]:
# ── Curvas de perda ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, title, tl, vl in zip(axes,
    ['Baseline (L=64, G=32, seq=60)', 'Exp.2 (L=128, G=64, seq=60)', 'Exp.3 (L=128, G=64, seq=90)'],
    [tl1, tl2, tl3], [vl1, vl2, vl3]):
    ax.plot(tl, label='Treino', lw=1.5)
    ax.plot(vl, label='Validacao', lw=1.5, ls='--')
    ax.set_title(title, fontsize=10); ax.set_xlabel('Epoca'); ax.set_ylabel('MSE Loss')
    ax.set_yscale('log'); ax.legend(fontsize=9); ax.grid(True, alpha=0.4)
plt.suptitle('Curvas de Perda — 3 Experimentos', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Real vs Previsto — Melhor modelo (Exp.3) ───────────────────
test_dates = data.index[split:]
cr = ['#1f77b4', '#2ca02c', '#d62728', '#9467bd']
cp = ['#aec7e8', '#98df8a', '#ff9896', '#c5b0d5']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()
for i, (name, ax) in enumerate(zip(NAMES, axes)):
    ci = TARGET_COLS[i]
    real = inv(y_te3[:, i], ci)
    prev = inv(preds3[:, i], ci)
    ax.plot(test_dates, real, label='Real',     color=cr[i], lw=1.8)
    ax.plot(test_dates, prev, label='Previsto', color=cp[i], lw=1.5, ls='--')
    ax.set_title(f'{name}\nR2={met3["R2"][name]:.4f}  RMSE=R${met3["RMSE (R$)"][name]:.2f}  MAPE={met3["MAPE (%)"][name]:.2f}%',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Data'); ax.set_ylabel('Preco (R$)')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('Preco Real vs Previsto — Exp.3 (LSTM=128, GRU=64, seq=90)',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Scatter: Previsto vs Real ──────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()
for i, (name, ax) in enumerate(zip(NAMES, axes)):
    ci   = TARGET_COLS[i]
    real = inv(y_te3[:, i], ci)
    prev = inv(preds3[:, i], ci)
    ax.scatter(real, prev, alpha=0.5, s=15, color=cr[i])
    mn, mx = min(real.min(), prev.min()), max(real.max(), prev.max())
    ax.plot([mn, mx], [mn, mx], 'k--', lw=1.2, label='Ideal')
    ax.set_title(f'{name}\nR2={met3["R2"][name]:.4f}  RMSE=R${met3["RMSE (R$)"][name]:.2f}',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Preco Real (R$)'); ax.set_ylabel('Preco Previsto (R$)')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.suptitle('Dispersao: Previsto vs Real (Melhor Modelo)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Previsao de 7 dias futuros ─────────────────────────────────
import datetime

SEQ_BEST = 90
m3.eval()
inp = torch.tensor(data_scaled[-SEQ_BEST:], dtype=torch.float32).unsqueeze(0).to(DEVICE)
future_norm = []

for _ in range(7):
    with torch.no_grad():
        pred = m3(inp).cpu().numpy()[0]
    future_norm.append(pred)
    step = inp[0, -1, :].cpu().numpy().copy()
    for k, ci in enumerate(TARGET_COLS):
        step[ci] = pred[k]
    inp = torch.cat([inp[:, 1:, :],
                     torch.tensor(step, dtype=torch.float32).view(1,1,-1).to(DEVICE)], dim=1)

future_norm  = np.array(future_norm)  # (7, 4)
future_prices = np.stack([inv(future_norm[:, k], TARGET_COLS[k]) for k in range(N_STOCKS)])

last_date    = data.index[-1]
future_dates = [last_date + datetime.timedelta(days=d) for d in range(1, 8)]

future_df = pd.DataFrame(future_prices.T, columns=TICKERS,
                         index=[d.strftime('%d/%m/%Y') for d in future_dates])
print('=== PREVISAO DE 7 DIAS (R$) ===')
print(future_df.round(2).to_string())

# Grafico historico + previsao
n_hist = 30
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()
for i, (t, name, ax) in enumerate(zip(TICKERS, NAMES, axes)):
    hist_vals  = data[f'{t}_Close'].values[-n_hist:]
    hist_dates = data.index[-n_hist:]
    x_conn = [hist_dates[-1]] + future_dates
    y_conn = [hist_vals[-1]]  + list(future_prices[i])
    ax.plot(hist_dates, hist_vals, color=cr[i], lw=1.8, label='Historico (30d)')
    ax.plot(x_conn, y_conn, color=cp[i], lw=1.8, ls='--', marker='o', ms=5, label='Previsao (+7d)')
    ax.axvline(hist_dates[-1], color='gray', ls=':', lw=1)
    ax.set_title(f'{name} ({t})', fontsize=10, fontweight='bold')
    ax.set_xlabel('Data'); ax.set_ylabel('Preco (R$)')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('Historico Recente + Previsao de 7 Dias Futuros', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()